<a href="https://colab.research.google.com/github/jioh-shin/econ5200-PS01/blob/main/Problem%20Set%201/Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [339]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [340]:
# Clean consumer transactions
n=20000
customer_id = rng.integers(1,1500,size=n)
year=rng.choice([1,2],size=n)
basket_value=rng.lognormal(mean=np.log(50),sigma=0.6,size=n)
np.median(basket_value)
df=pd.DataFrame({"customer_id":customer_id,"year":year,"basket_value":basket_value})

In [341]:
# The truth: the consumer average per year, recorded before contamination
true_consumer_average=df.groupby("year")["basket_value"].mean()
true_consumer_average

,basket_value
year,
1,60.353529
2,60.367007


In [342]:
# Contamination 1: B2B orders
b2b_n=300
b2b_customer_id=rng.integers(1501,1601,size=b2b_n)
b2b_year=rng.choice([1,2],size=b2b_n,p=[0.1,0.9])
b2b_basket_value=rng.lognormal(mean=np.log(200),sigma=0.6,size=b2b_n)
b2b_df=pd.DataFrame({"customer_id":b2b_customer_id,"year":b2b_year,"basket_value":b2b_basket_value})
df = pd.concat([df, b2b_df], ignore_index=True)

In [343]:
# Contamination 2: what gets logged changes between year 1 and year 2
cancel_n=250
cancel_customer_id=rng.integers(1,1501,size=cancel_n)
cancel_year=np.full(cancel_n,2)
cancel_basket_value=rng.lognormal(mean=np.log(85),sigma=0.6,size=cancel_n)
cancel_df=pd.DataFrame({"customer_id":cancel_customer_id,"year":cancel_year,"basket_value":cancel_basket_value})
df = pd.concat([df, cancel_df], ignore_index=True)

### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [344]:
# The dashboard's number vs the truth
dashboard_number=df.groupby("year")["basket_value"].mean()
gap=dashboard_number-true_consumer_average
dashboard_number, gap

(year
 1    60.933278
 2    65.763605
 Name: basket_value, dtype: float64,
 year
 1    0.579750
 2    5.396598
 Name: basket_value, dtype: float64)

---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [345]:
# How much of the gap is the skew and the B2B tail?
no_b2b_df = df[df["customer_id"] <= 1500]
no_b2b_mean = no_b2b_df.groupby("year")["basket_value"].mean()
no_b2b_mean
dashboard_change = dashboard_number.loc[2] - dashboard_number.loc[1]
no_b2b_change = no_b2b_mean.loc[2] -no_b2b_mean.loc[1]
dashboard_change,no_b2b_change
b2b_contribution=dashboard_change-no_b2b_change
b2b_contribution


np.float64(3.895557630863827)

In [346]:
# How much is the change in what was logged?
no_cancel_df = df.iloc[:-cancel_n]
no_cancel_mean = no_cancel_df.groupby("year")["basket_value"].mean()
no_cancel_mean
no_cancel_change=no_cancel_mean.loc[2]-no_cancel_mean.loc[1]
cancel_contribution=dashboard_change-no_cancel_change
cancel_contribution

np.float64(0.7889544281529766)

| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail |3.8956 |Removed B2B transactions and recalculated the change |
| Change in what was logged |0.7890 |Removed the newly logged cancelled transactions and recalculated the change  |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [347]:
# Median, trimmed mean, and the mean after your B2B exclusion rule
median_by_year = df.groupby("year")["basket_value"].median()
year1=df[df["year"] == 1]["basket_value"]
year2=df[df["year"] == 2]["basket_value"]
trimmed_mean_year1=stats.trim_mean(year1, 0.1)
trimmed_mean_year2=stats.trim_mean(year2, 0.1)
trimmed_mean_year1, trimmed_mean_year2
no_b2b_mean=no_b2b_df.groupby("year")["basket_value"].mean()
median_pct = (median_by_year.loc[2] / median_by_year.loc[1] - 1) * 100
trimmed_pct = (trimmed_mean_year2 / trimmed_mean_year1 - 1) * 100
no_b2b_pct = (no_b2b_mean.loc[2] / no_b2b_mean.loc[1] - 1) * 100
median_pct, trimmed_pct, no_b2b_pct

(np.float64(3.71003181346099),
 np.float64(4.391990252667699),
 np.float64(1.548822419641005))

| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median |3.71% |The typical transaction is the quantity of interest. |If changes in large baskets are economically meaningful. |
| Trimmed mean |4.39% |Extreme values are mostly noise or contamination. |If extreme values are legitimate transactions that should be included. |
| Mean after B2B exclusion |1.55% |The dashboard is intended to measure consumer basket value, not B2B orders. |If B2B transactions are actually part of the target population. |

**Which number should the dashboard show?** Commit to one and defend it.

I would use the mean after B2B exclusion, which shows a 1.55% increase. Since the dashboard is supposed to show consumer basket value, I think B2B orders should be excluded. This gives a better estimate of the change in consumer spending.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [348]:
import os
os.makedirs("src", exist_ok=True)

In [349]:
%%writefile src/basket_metrics.py
"""Functions for checking data and calculating robust means"""

import numpy as np
import pandas as pd


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: missingness, dtype, skew and outlier count."""
    report = pd.DataFrame()
    report["missingness"] = df.isna().sum()
    report["dtype"] = df.dtypes
    report["skew"] = df.skew(numeric_only=True)
    outlier_counts = {}
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1
            outlier_counts[col] = ((df[col] < q1 - 1.5 * iqr) |
                                   (df[col] > q3 + 1.5 * iqr)).sum()
        else:
            outlier_counts[col] = np.nan
    report["outlier_count"] = pd.Series(outlier_counts)
    return report


def robust_mean(x, method: str = "median", customer_id=None) -> float:
    """Calculate a robust average using median, trimming, or B2B exclusion."""
    if method == "median":
        return float(np.median(x))

    elif method == "trimmed":
        x_sorted = np.sort(x)
        n = len(x_sorted)
        cut = int(n * 0.1)
        return float(np.mean(x_sorted[cut:n-cut]))

    elif method == "b2b_exclusion":
        if customer_id is None:
            raise ValueError("customer_id is required for B2B exclusion")
        x = np.asarray(x)
        customer_id = np.asarray(customer_id)
        return float(np.mean(x[customer_id <= 1500]))

    else:
        raise ValueError("method must be 'median','trimmed',or 'b2b_exclusion")


if __name__ == "__main__":
    # A short demonstration that runs when the file is run as a script
    x = np.array([10, 20, 30, 40, 100])
    print("Median:", robust_mean(x, method="median"))
    print("Trimmed mean:", robust_mean(x, method="trimmed"))

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [350]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)     # picks up your edits when you re-run %%writefile

# At least three properties that must hold, for example:
# assert abs(basket_metrics.robust_mean(symmetric_data) - symmetric_data.mean()) < tolerance
test_data = np.array([1, 2, 3, 4])
assert basket_metrics.robust_mean(test_data, method="median") == 2.5
trimmed_data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 100])
assert basket_metrics.robust_mean(trimmed_data, method="trimmed") == 5.5
outlier_data = np.array([1, 2, 3, 4, 1000])
assert basket_metrics.robust_mean(outlier_data, method="median") == 3.0
b2b_values = np.array([10, 20, 30, 100])
customer_ids = np.array([100, 500, 1500, 1501])
assert basket_metrics.robust_mean(b2b_values,method="b2b_exclusion",customer_id=customer_ids) == 20.0

---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:**

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**

In [351]:
# The analysis that settles it
# Compare consumer and B2B transactions

consumer = df[df["customer_id"] <= 1500]
b2b = df[df["customer_id"] > 1500]

comparison = pd.DataFrame({
    "transaction_count": [len(consumer), len(b2b)],
    "average_basket": [
        consumer["basket_value"].mean(),
        b2b["basket_value"].mean()
    ],
    "median_basket": [
        consumer["basket_value"].median(),
        b2b["basket_value"].median()
    ],
    "revenue_share": [
        consumer["basket_value"].sum() / df["basket_value"].sum(),
        b2b["basket_value"].sum() / df["basket_value"].sum()
    ]
}, index=["Consumer", "B2B"])

comparison

,transaction_count,average_basket,median_basket,revenue_share
Consumer,20250,60.830469,50.561874,0.945077
B2B,300,238.624979,208.108642,0.054923


**What you found, including if it changes your Phase 2 recommendation:**

The strongest objection was whether B2B transactions should actually be excluded. Since B2B transactions are still real sales, I think excluding them only makes sense if the dashboard is supposed to measure consumer basket value. To check this, I compared B2B and consumer transactions separately. There were 20,250 consumer transactions and only 300 B2B transactions, but the average B2B basket was much higher, at 238.62 dollars compared with 60.80 dollars for consumers. B2B transactions made up about 5.5% of total revenue. This suggests that B2B transactions are a small but very different group that can affect the overall mean. This does not change my Phase 2 recommendation, so I would still use the mean after B2B exclusion if the dashboard is supposed to measure consumer basket value.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

The dashboard made the increase in average basket value look bigger than it actually was. After excluding B2B transactions, the mean shows a 1.55% increase from Year 1 to Year 2. I am fairly confident in this result because there were only a small number of B2B transactions, but their average basket value was much higher than consumer transactions. Before fully relying on this result, I would still want to check if the B2B rule correctly separates B2B transactions from regular consumer transactions.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors